# Matrix Reconstruction (Load Trained Model)
Load a saved Linear AE or AE from disk/Drive, recreate the model from the results JSON, analyze latent distributions, and compute reconstruction metrics (MSE, MAE, Frobenius).

In [235]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Step 1: Load the desired dataset (test.pt / all.pt)
Load correlation matrices.

In [236]:
FILE_NAME = 'data_00_20'
WINDOW_SIZE = 252
STRIDE = 5
DATASET_NAME = f'{FILE_NAME}_w{WINDOW_SIZE}_s{STRIDE}'
RUN = 'linearAE_150dim_0001'
FORWARD_DAYS = 21
RESULTS_JSON_PATH = f'models/{DATASET_NAME}/{FORWARD_DAYS}_days_gap/linearAE/{RUN}/run_results.json'


if 'google.colab' in sys.modules:
    print('Environment: Google Colab')
    IS_COLAB = True
else:
    print('Environment: Local PC')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    base_dir = DRIVE_ROOT / 'dataset_tesi'
else:
    project_root = Path.cwd().resolve().parent
    DRIVE_ROOT = None
    base_dir = project_root / 'data' / 'processed' / FILE_NAME / 'dataset' / DATASET_NAME / f'{FORWARD_DAYS}_days_gap'

dataset_dir = base_dir

DATASET_ORDER = ['train', 'val', 'test', 'all']
DATASET_FILES = {}
for name in DATASET_ORDER:
    candidate = dataset_dir / f'{name}.pt'
    if candidate.exists():
        DATASET_FILES[name] = candidate

if not DATASET_FILES:
    raise FileNotFoundError(
        f'No dataset .pt files found in: {dataset_dir.absolute()}'
    )

print(f'Dataset selected: {DATASET_NAME}')
print(f'Available datasets: {", ".join(DATASET_FILES.keys())}')

Environment: Local PC
Dataset selected: data_00_20_w252_s5
Available datasets: train, val, test, all


In [237]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        indices = payload.get('indices', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        indices = None
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), indices, meta


SAMPLE_DATASET = next(iter(DATASET_FILES))
sample_corr, sample_indices, sample_meta = load_corr_payload(DATASET_FILES[SAMPLE_DATASET])

print(f'Sample dataset: {SAMPLE_DATASET}')
print(f'Correlation tensor shape: {sample_corr.shape}')
if sample_indices is not None:
    print(f'Indices: {sample_indices[:10]}')
else:
    print('Indices: not found in payload')

Sample dataset: train
Correlation tensor shape: torch.Size([679, 100, 100])
Indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Step 2: Prepare Matrices (Full Dataset)
Use full correlation matrices as input. Each matrix is flattened to a vector of size `N x N`.

In [238]:
def prepare_inputs(corr_tensor: torch.Tensor):
    all_np = corr_tensor.numpy().astype(np.float32)
    n_matrices, n_assets, _ = all_np.shape

    n_features = n_assets * (n_assets - 1) // 2
    tril_idx = np.tril_indices(n_assets, k=-1)
    extracted_np = all_np[:, tril_idx[0], tril_idx[1]]
    x_all = torch.from_numpy(extracted_np)

    return all_np, x_all, n_assets, n_features, tril_idx


sample_all_np, sample_x_all, N_ASSETS, N_FEATURES, SAMPLE_TRIL_IDX = prepare_inputs(sample_corr)

print(f"{'='*40}")
print(f"Number of matrices   : {sample_all_np.shape[0]}")
print(f"Original matrix shape: ({N_ASSETS}, {N_ASSETS})")
print(f"Flattened input size : {N_FEATURES}")
print(f"Sample tensor shape  : {tuple(sample_x_all.shape)}")
print(f"{'='*40}")

Number of matrices   : 679
Original matrix shape: (100, 100)
Flattened input size : 4950
Sample tensor shape  : (679, 4950)


## Step 3: Define the Models
Use a Linear Autoencoder (no activations) or a non-linear AutoEncoder (with hidden layers).

In [239]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dims=None, dropout_prob: float = 0.02):
        super().__init__()

        # Se hidden_dims è None, creiamo un Linear AE semplice
        if hidden_dims is None:
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, latent_dim, bias=False),
            )
            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, input_dim, bias=False),
            )
        else:
            # Caso Deep Autoencoder con Dropout e attivazioni
            dimensions = [input_dim, *hidden_dims, latent_dim]

            # --- ENCODER ---
            encoder_layers = []
            for i in range(len(dimensions) - 1):
                encoder_layers.append(nn.Linear(dimensions[i], dimensions[i + 1]))
                if i < len(dimensions) - 2:
                    encoder_layers.append(nn.LeakyReLU(0.01))
                    encoder_layers.append(nn.Dropout(dropout_prob))
            self.encoder = nn.Sequential(*encoder_layers)

            # --- DECODER ---
            decoder_dims = dimensions[::-1]
            decoder_layers = []
            for i in range(len(decoder_dims) - 1):
                decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
                if i < len(decoder_dims) - 2:
                    decoder_layers.append(nn.LeakyReLU(0.01))
                else:
                    decoder_layers.append(nn.Tanh())
            self.decoder = nn.Sequential(*decoder_layers)

    def architecture_signature(self):
        return [
            [int(layer.in_features), int(layer.out_features)]
            for layer in self.encoder
            if isinstance(layer, nn.Linear)
        ]

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

## Step 4: Load Best Model from Results JSON
Select the results JSON path, load model details, recreate the architecture, and load weights.

In [240]:
def resolve_path(path_str, base_dir=None, results_dir=None):
    if path_str is None:
        return None
    p = Path(path_str)
    if p.is_absolute() and p.exists():
        return p.resolve()

    if results_dir is not None:
        candidate = results_dir / p
        if candidate.exists():
            return candidate.resolve()

    if base_dir is not None:
        candidate = base_dir / p
        if candidate.exists():
            return candidate.resolve()

    normalized = str(path_str).replace('\\', '/')
    if base_dir is not None and 'best_models_tesi' in normalized:
        rel = normalized[normalized.index('best_models_tesi'):]
        candidate = base_dir / rel
        if candidate.exists():
            return candidate.resolve()

    return p


results_path = Path(RESULTS_JSON_PATH)
if not results_path.is_absolute():
    if IS_COLAB:
        if DRIVE_ROOT is None:
            raise ValueError('DRIVE_ROOT is not set for Colab')
        results_path = (DRIVE_ROOT / results_path).resolve()
    else:
        results_path = (project_root / results_path).resolve()

if not results_path.exists():
    raise FileNotFoundError(f'Results JSON not found: {results_path}')

OUTPUT_DIR_NAME = 'analysis_outputs'
analysis_dir = results_path.parent / OUTPUT_DIR_NAME
analysis_dir.mkdir(parents=True, exist_ok=True)

with open(results_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

model_cfg = results.get('model', {})
latent_dim = int(model_cfg.get('latent_dim', 0))
if latent_dim <= 0:
    raise ValueError('Invalid latent_dim in results JSON')

hidden_dims = model_cfg.get('hidden_dims', None)
model_type_raw = str(model_cfg.get('model_type', '')).strip().lower()
if model_type_raw in {'linear', 'linearae', 'linear_ae', 'linear-ae'}:
    model_type = 'linearAE'
elif model_type_raw in {'ae', 'autoencoder', 'auto'}:
    model_type = 'AE'
else:
    model_type = 'AE' if hidden_dims is not None else 'linearAE'

if model_type == 'linearAE':
    hidden_dims = None
else:
    if hidden_dims is None:
        raise ValueError('hidden_dims missing for AE in results JSON')

input_dim = int(model_cfg.get('input_dim', N_FEATURES))
if input_dim != N_FEATURES:
    raise ValueError(f'Input dim mismatch: json={input_dim}, data={N_FEATURES}')

dropout_prob = model_cfg.get('dropout', 0.02)
if dropout_prob is None:
    dropout_prob = 0.0
dropout_prob = float(dropout_prob)

model = AutoEncoder(
    input_dim=input_dim,
    latent_dim=latent_dim,
    hidden_dims=hidden_dims,
    dropout_prob=dropout_prob,
).to(device)

weights_path = results.get('artifacts', {}).get('best_model_path', None)

if weights_path is None:
    run_name = results.get('run', 'run')
    if model_type == 'linearAE':
        weights_path = f'linear_AE_best_{run_name}.pt'
    else:
        weights_path = f'AE_best_{run_name}.pt'

base_dir = DRIVE_ROOT if IS_COLAB else project_root
weights_path = resolve_path(weights_path, base_dir=base_dir, results_dir=results_path.parent)
if weights_path is None or not weights_path.exists():
    raise FileNotFoundError(f'Weights file not found: {weights_path}')

checkpoint = torch.load(weights_path, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

print(f'Model type: {model_type} | latent_dim={latent_dim} | input_dim={input_dim}')
print(f'Loaded weights: {weights_path}')

FileNotFoundError: Results JSON not found: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_150dim_0001\run_results.json

In [ ]:
print(model)

AutoEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=4950, out_features=100, bias=False)
  )
  (decoder): Sequential(
    (0): Linear(in_features=100, out_features=4950, bias=False)
  )
)


## Step 5: Latent Space Analysis + Reconstruction Errors Analysis
Encode the matrices into the latent space and analyze feature distributions + Analyse MSE, MAE and Frobenius.

In [ ]:
# 1. Funzione di ricostruzione con gestione automatica del device
def compute_reconstruction(model: nn.Module, x_tensor: torch.Tensor, batch_size: int = 64, device=None):
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    latents = []
    reconstructions = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            z = model.encoder(xb)
            latents.append(z.cpu().numpy())
            recon = model.decoder(z)
            reconstructions.append(recon.cpu().numpy())

    return np.concatenate(latents, axis=0), np.concatenate(reconstructions, axis=0)

# 2. Funzione per gli errori (rimane invariata, ora riceverà matrici quadrate corrette)
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    if original.shape != reconstructed.shape:
        raise ValueError(f'Input shapes mismatch: {original.shape} vs {reconstructed.shape}')

    diff = original - reconstructed

    if original.ndim == 3:
        mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
        mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
        fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

        summary = pd.DataFrame({
            'MSE': mse_per_matrix,
            'MAE': mae_per_matrix,
            'Frobenius': fro_per_matrix,
        })
    elif original.ndim == 2:
        mse_per_vec = np.mean(diff ** 2, axis=1)
        mae_per_vec = np.mean(np.abs(diff), axis=1)
        l2_per_vec = np.linalg.norm(diff, axis=1)

        summary = pd.DataFrame({
            'MSE': mse_per_vec,
            'MAE': mae_per_vec,
            'L2': l2_per_vec,
        })
    else:
        raise ValueError(f'Expected 2D or 3D arrays, got ndim={original.ndim}')

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [ ]:
# ==========================================
# ESECUZIONE DEL CALCOLO E RICOSTRUZIONE
# ==========================================


def process_dataset(dataset_name: str, matrix_file: Path):
    corr, indices, meta = load_corr_payload(matrix_file)
    all_np, x_all, n_assets, n_features, tril_idx = prepare_inputs(corr)

    if n_features != N_FEATURES:
        raise ValueError(
            f'Input dim mismatch for {dataset_name}: expected {N_FEATURES}, got {n_features}'
        )

    latents_all, recon_flat = compute_reconstruction(model, x_all, batch_size=64)

    n_matrices = all_np.shape[0]
    recon_all = np.zeros((n_matrices, n_assets, n_assets), dtype=np.float32)

    recon_all[:, tril_idx[0], tril_idx[1]] = recon_flat
    recon_all[:, tril_idx[1], tril_idx[0]] = recon_flat

    diag_idx = np.arange(n_assets)
    recon_all[:, diag_idx, diag_idx] = 1.0

    errors_df, summary_df = reconstruction_errors(all_np, recon_all)

    latent_dim = latents_all.shape[1]
    latent_cols = [f'z{i + 1}' for i in range(latent_dim)]
    latent_df = pd.DataFrame(latents_all, columns=latent_cols)

    if indices is None:
        indices = np.arange(len(errors_df))

    if len(indices) != len(errors_df):
        raise ValueError(f'Error rows ({len(errors_df)}) do not match indices ({len(indices)})')
    errors_df.insert(0, 'matrix_idx', indices)

    print(f'\nReconstruction error summary ({dataset_name} data):')
    display(summary_df)

    errors_json_path = analysis_dir / f'reconstruction_errors_{dataset_name}_{RUN}.json'
    summary_json_path = analysis_dir / f'reconstruction_summary_{dataset_name}_{RUN}.json'
    errors_df.to_json(errors_json_path, orient='records', indent=2)
    summary_df.to_json(summary_json_path, orient='records', indent=2)

    print(f'Saved per-matrix errors: {errors_json_path}')
    print(f'Saved summary stats: {summary_json_path}')

    original_payload = torch.load(matrix_file, map_location='cpu')
    if isinstance(original_payload, dict):
        extended_payload = original_payload.copy()
    else:
        extended_payload = {'corr_tensor': original_payload}

    recon_tensor = torch.from_numpy(recon_all).float()
    extended_payload['corr_tensor_reconstructed'] = recon_tensor

    tickers = None
    if isinstance(meta, dict):
        tickers = meta.get('meta', {}).get('tickers', None)
    if tickers is not None:
        extended_payload['tickers'] = tickers

    output_path = analysis_dir / f'{dataset_name}_reconstructed_{RUN}.pt'
    torch.save(extended_payload, output_path)
    print(f'Saved reconstructed matrices: {output_path}')

    print(f'\nLatent distribution summary ({dataset_name} data):')
    display(latent_df.describe().T)

    valid_cols = [
        col for col in latent_cols
        if latent_df[col].notna().any() and latent_df[col].nunique() > 1
    ]
    latent_df = latent_df[valid_cols]

    if len(indices) != len(latent_df):
        raise ValueError(f'Latent rows ({len(latent_df)}) do not match indices ({len(indices)})')
    latent_df.insert(0, 'matrix_idx', indices)

    latent_json_path = analysis_dir / f'latent_{dataset_name}_{RUN}.json'
    latent_df.to_json(latent_json_path, orient='records', indent=2)
    print(f'Saved latent samples: {latent_json_path}')

    if len(valid_cols) < 2:
        print('Not enough valid latent dimensions for pairwise plots.')
    elif len(valid_cols) > 20:
        print(f'Skipping pairwise plot: too many latent dimensions ({len(valid_cols)} > 20).')
    else:
        plot_df = latent_df[valid_cols]
        grid = sns.PairGrid(plot_df, corner=True, diag_sharey=False)
        grid.map_lower(sns.scatterplot, s=12, alpha=0.6, color='#2a9d8f')
        grid.figure.suptitle(
            f'Pairwise latent dimension plots ({dataset_name} data)',
            y=1.02,
        )
        pairplot_path = analysis_dir / f'latent_pairwise_{dataset_name}_{RUN}.png'
        grid.savefig(pairplot_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved latent pairwise plot: {pairplot_path}')


for dataset_name, matrix_file in DATASET_FILES.items():
    print(f'\n=== Processing {dataset_name} ({matrix_file.name}) ===')
    process_dataset(dataset_name, matrix_file)


=== Processing train (train.pt) ===

Reconstruction error summary (train data):


,mean,std,min,median,max
MSE,0.000076,0.000032,0.000016,0.000070,0.000200
MAE,0.006519,0.001316,0.003068,0.006445,0.010355
Frobenius,0.851161,0.177729,0.394785,0.838376,1.415163


Saved per-matrix errors: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\reconstruction_errors_train_linearAE_100dim_0001.json
Saved summary stats: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\reconstruction_summary_train_linearAE_100dim_0001.json
Saved reconstructed matrices: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\train_reconstructed_linearAE_100dim_0001.pt

Latent distribution summary (train data):


,count,mean,std,min,25%,50%,75%,max
z1,679.0,-0.000746,0.271756,-0.694766,-0.165222,0.008667,0.162908,0.884699
z2,679.0,-0.010985,0.341792,-0.936113,-0.267136,0.034521,0.217889,0.755663
z3,679.0,-0.021543,0.351447,-0.841317,-0.276063,-0.022452,0.283283,0.687137
z4,679.0,0.039202,0.591428,-0.929575,-0.386730,-0.068885,0.257993,1.590633
z5,679.0,-0.004893,0.360988,-0.889350,-0.209739,-0.010328,0.271411,0.857709
...,...,...,...,...,...,...,...,...
z96,679.0,0.004302,0.340892,-0.897642,-0.297025,-0.009598,0.289239,0.695969
z97,679.0,-0.015083,0.297849,-0.725919,-0.215500,0.006999,0.203483,0.670305
z98,679.0,0.001198,0.134955,-0.412339,-0.088857,-0.002163,0.080537,0.464631
z99,679.0,0.031427,0.607750,-1.675032,-0.315017,0.124863,0.397737,1.410676


Saved latent samples: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\latent_train_linearAE_100dim_0001.json
Skipping pairwise plot: too many latent dimensions (100 > 20).

=== Processing val (val.pt) ===

Reconstruction error summary (val data):


...

Saved latent samples: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\latent_test_linearAE_100dim_0001.json
Skipping pairwise plot: too many latent dimensions (100 > 20).

=== Processing all (all.pt) ===

Reconstruction error summary (all data):


,mean,std,min,median,max
MSE,0.002569,0.004098,0.000016,0.000087,0.014072
MAE,0.025734,0.030044,0.003068,0.007172,0.091570
Frobenius,3.306841,3.843560,0.394785,0.933662,11.862623


Saved per-matrix errors: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\reconstruction_errors_all_linearAE_100dim_0001.json
Saved summary stats: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\reconstruction_summary_all_linearAE_100dim_0001.json
Saved reconstructed matrices: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\all_reconstructed_linearAE_100dim_0001.pt

Latent distribution summary (all data):


,count,mean,std,min,25%,50%,75%,max
z1,980.0,-0.145894,0.379211,-1.240462,-0.338289,-0.086829,0.097329,0.884699
z2,980.0,-0.012355,0.316227,-0.936113,-0.233078,0.014333,0.200748,0.755663
z3,980.0,-0.038036,0.409164,-0.841317,-0.343842,-0.090155,0.239816,1.239491
z4,980.0,-0.149048,0.587081,-0.956050,-0.592473,-0.233744,0.111908,1.590633
z5,980.0,-0.074681,0.390324,-1.178831,-0.323532,-0.047851,0.193745,0.857709
...,...,...,...,...,...,...,...,...
z96,980.0,0.023614,0.306853,-0.897642,-0.218797,0.032608,0.235619,0.719737
z97,980.0,-0.097643,0.336684,-1.079245,-0.335489,-0.080638,0.165359,0.670305
z98,980.0,-0.048754,0.262651,-0.859314,-0.140585,-0.013692,0.102976,0.508315
z99,980.0,0.001076,0.534549,-1.675032,-0.292496,0.039207,0.311162,1.410676


Saved latent samples: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\linearAE\linearAE_100dim_0001\analysis_outputs\latent_all_linearAE_100dim_0001.json
Skipping pairwise plot: too many latent dimensions (100 > 20).
